In [ ]:
# WITHIN-SUBJECT: sessões 1-4 = treino, sessões 5-7 = teste
import mne
import torch
from moabb.datasets import Zhou2020

dataset = Zhou2020()
data = dataset.get_data()

subjects = list(data.keys())
N_SESSOES_TREINO = 4
N_SESSOES_TESTE = 3

def epochs_from_session(session_dict, tmin=0, tmax=4.0):

    raws = list(session_dict.values())
    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# dicionário: subject_id -> {"train": epochs_sessao0, "test": epochs_sessao1}
within_subject_tensors = {}

for subj in subjects:
    session_keys = list(data[subj].keys())

    primeira_epochs = epochs_from_session(data[subj][session_keys[0]])
    event_id_fixo = primeira_epochs.event_id
    
    sessoes_treino = session_keys[:N_SESSOES_TREINO]        # sessões 1, 2, 3,4
    sessoes_teste = session_keys[:N_SESSOES_TESTE]  # sessões 5,6,7
    
    train_list = [primeira_epochs] + [
        epochs_from_session(data[subj][k], event_id=event_id_fixo)
        for k in sessoes_treino[1:]
    ]
    test_list = [
        epochs_from_session(data[subj][k], event_id=event_id_fixo)
        for k in sessoes_teste
    ]
    
    train_epochs = mne.concatenate_epochs(train_list)
    test_epochs = mne.concatenate_epochs(test_list)
    
    X_tr = train_epochs.get_data() * 1e6
    y_tr = train_epochs.events[:, -1]
    X_te = test_epochs.get_data() * 1e6
    y_te = test_epochs.events[:, -1]

    min_lbl = y_tr.min()

    within_subject_tensors[subj] = {
        "X_train": torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_tr - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_te, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_te - min_lbl, dtype=torch.long),
    }

for subj, tensors in within_subject_tensors.items():
    X_train_sub = tensors["X_train"]
    y_train_sub = tensors["y_train"]
    X_test_sub = tensors["X_test"]
    y_test_sub = tensors["y_test"]

In [ ]:
#CROS-SUBJECT
# múltiplas combinações aleatórias de 80% treino / 20% teste (por sujeito)
import mne
import torch
import numpy as np
from moabb.datasets import Zhou2020
from sklearn.model_selection import ShuffleSplit


dataset = Zhou2020()
data = dataset.get_data()

subjects = list(data.keys())


def epochs_from_subject(session_dict, tmin=0, tmax=4.0):
   
    raws = []
    for sess in session_dict.values():
        raws.extend(list(sess.values()))

    raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')
    return epochs


# Extrai os epochs de cada sujeito 
subject_epochs = {}
event_id_fixo = None
for subj in subjects:
    ep = epochs_from_subject(data[subj], event_id=event_id_fixo)
    if event_id_fixo is None:
        event_id_fixo = ep.event_id  # fixado a partir do primeiro sujeito
    subject_epochs[subj] = ep


all_events = np.concatenate([subject_epochs[s].events[:, -1] for s in subjects])
min_lbl = all_events.min()

N_SPLITS = 7        # quantas combinações diferentes de 80/20 gerar
TEST_SIZE = 0.2      # fração de sujeitos reservada para teste em cada combinação
RANDOM_STATE = 42    # fixa a semente para reprodutibilidade

# Monta um fold para cada sujeito deixado de fora (leave-one-subject-out)
splitter = ShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
subjects_arr = np.array(subjects)
cross_subject_folds = {}

for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(subjects_arr)):
    train_subjs = subjects_arr[train_idx].tolist()
    test_subjs = subjects_arr[test_idx].tolist()

    X_train_list, y_train_list = [], []
    for s in train_subjs:
        ep = subject_epochs[s]
        X_train_list.append(ep.get_data() * 1e6)
        y_train_list.append(ep.events[:, -1])

    X_test_list, y_test_list = [], []
    for s in test_subjs:
        ep = subject_epochs[s]
        X_test_list.append(ep.get_data() * 1e6)
        y_test_list.append(ep.events[:, -1])

    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)
    X_test = np.concatenate(X_test_list, axis=0)
    y_test = np.concatenate(y_test_list, axis=0)

    cross_subject_folds[fold_idx] = {
        "train_subjects": train_subjs,
        "test_subjects": test_subjs,
        "X_train": torch.tensor(X_train, dtype=torch.float32).unsqueeze(1),
        "y_train": torch.tensor(y_train - min_lbl, dtype=torch.long),
        "X_test": torch.tensor(X_test, dtype=torch.float32).unsqueeze(1),
        "y_test": torch.tensor(y_test - min_lbl, dtype=torch.long),
    }


# iterar sobre os folds, cada um deixando um sujeito de fora
for fold_idx, tensors in cross_subject_folds.items():
    X_train_fold = tensors["X_train"]
    y_train_fold = tensors["y_train"]
    X_test_fold = tensors["X_test"]
    y_test_fold = tensors["y_test"]

    print(f"\nCombinação {fold_idx}:")
    print(f"  Treino ({len(tensors['train_subjects'])} sujeitos): {tensors['train_subjects']}")
    print(f"  Teste  ({len(tensors['test_subjects'])} sujeitos): {tensors['test_subjects']}")
    print(f"  X_train={tuple(X_train_fold.shape)}, X_test={tuple(X_test_fold.shape)}")